In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("saurabhshahane/bangla-wikipedia")

print("Path to dataset files:", path)

100%|██████████| 71.2M/71.2M [00:01<00:00, 71.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/saurabhshahane/bangla-wikipedia/versions/1


In [2]:
import tensorflow as tf
import numpy as np
import pandas as pd
import re
from tensorflow.keras.layers import (
    TextVectorization, Layer, Dense, LayerNormalization,
    MultiHeadAttention, Dropout, Embedding, Input
)
from tensorflow.keras.models import Model

In [4]:
import glob
import os


print("\nFiles inside dataset folder:")
for f in os.listdir(path):
    print(" -", f)


csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print("\nCSV files found:", csv_files)

CSV_PATH = csv_files[0] if csv_files else None
print("\nUsing CSV_PATH:", CSV_PATH)


Files inside dataset folder:
 - wiki.csv

CSV files found: ['/root/.cache/kagglehub/datasets/saurabhshahane/bangla-wikipedia/versions/1/wiki.csv']

Using CSV_PATH: /root/.cache/kagglehub/datasets/saurabhshahane/bangla-wikipedia/versions/1/wiki.csv


In [5]:
df = pd.read_csv(CSV_PATH)
print("Columns found:", list(df.columns))
print(df.head())

Columns found: ['id', 'text', 'title', 'url']
       id                                               text  \
0    2341  বংশী বাংলাদেশের একটি ক্ষুদ্রতম উপজাতি। এরা টাঙ...   
1  107282  আবহাওয়াবিদ্যা (বা আবহবিদ্যা) মানে এককথায় আবহ...   
2  107838  কোরিয়ার ওয়ার্কার্স পার্টি হল গণতান্ত্রিক গণপ...   
3    4542  মোহাম্মদ মোস্তফা কামাল (১৬ ডিসেম্বর ১৯৪৭ - এপ্...   
4    4626  মণিপুরী সংস্কৃতির উজ্জ্বলতম দিক হলো মণিপুরী নৃ...   

                         title                                         url  
0                         বংশী    https://bn.wikipedia.org/wiki?curid=2341  
1               আবহাওয়াবিদ্যা  https://bn.wikipedia.org/wiki?curid=107282  
2  কোরিয়ার ওয়ার্কার্স পার্টি  https://bn.wikipedia.org/wiki?curid=107838  
3   মোস্তফা কামাল (বীরশ্রেষ্ঠ)    https://bn.wikipedia.org/wiki?curid=4542  
4              মণিপুরী (নৃত্য)    https://bn.wikipedia.org/wiki?curid=4626  


In [6]:
possible_cols = ["text", "content", "article", "body", "articleText"]
text_col = None
for col in possible_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    # কোনোটাই না মিললে, সবচেয়ে বেশি average character length যে column-এ
    # সেটাকেই ধরে নেওয়া হচ্ছে main text column হিসেবে
    text_col = max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())

print("Using text column:", text_col)

Using text column: text


In [40]:
NUM_ARTICLES = 300   # কতগুলো article নিয়ে train করবেন, দরকার মতো বাড়ান/কমান
MAX_CHARS_PER_ARTICLE = 800  # প্রতিটা article থেকে কতটুকু text রাখবেন

subset = df[text_col].dropna().astype(str).head(NUM_ARTICLES)
subset = subset.apply(lambda t: t[:MAX_CHARS_PER_ARTICLE])

In [41]:
def clean_bangla_text(t):
    t = re.sub(r"\[\d+\]", " ", t)          # citation বর্গ বন্ধনী [1] সরানো
    t = re.sub(r"[a-zA-Z0-9]+", " ", t)      # ইংরেজি শব্দ/সংখ্যা সরানো (চাইলে রাখতে পারেন)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def split_sentences(t):
    # বাংলা দাড়ি (।) এবং সাধারণ পাংচুয়েশন দিয়ে বাক্যে ভাগ করা
    parts = re.split(r"[।!?]", t)
    return [p.strip() for p in parts if len(p.strip()) > 3]

text_lines = []
for article in subset:
    cleaned = clean_bangla_text(article)
    text_lines.extend(split_sentences(cleaned))

print("Total sentences collected:", len(text_lines))
print("Sample sentences:", text_lines[:5])

text = text_lines

Total sentences collected: 1913
Sample sentences: ['বংশী বাংলাদেশের একটি ক্ষুদ্রতম উপজাতি', 'এরা টাঙ্গাইল জেলার "মহানান্দপুর" এবং "দন্দোনিয়া" নামে পাশাপাশি দুইটি গ্রামের বসবাস করে', 'তারা নিজেদেরকে "সূর্য-বংশী" বলে থাকে', 'আবহাওয়াবিদ্যা (বা আবহবিদ্যা) মানে এককথায় আবহাওয়া সম্পর্কিত বিজ্ঞানকে আবহাওয়াবিদ্যা বলে', 'কোরিয়ার ওয়ার্কার্স পার্টি হল গণতান্ত্রিক গণপ্রজাতন্ত্রী কোরিয়া বা উত্তর কোরিয়ার শাসক রাজনৈতিক দল']


In [42]:
vectorizer = TextVectorization(
    standardize=None,  # বাংলার জন্য lower_and_strip_punctuation দরকার নেই (case নেই)
    output_mode="int",
)
vectorizer.adapt(text)
vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)
print("Vocabulary Size:", vocab_size)
print(vocab[:20])

index_to_word = dict(enumerate(vocab))
word_to_index = {word: i for i, word in enumerate(vocab)}

Vocabulary Size: 8200
['', '[UNK]', np.str_('এবং'), np.str_('ও'), np.str_('এই'), np.str_('একটি'), np.str_('হয়'), np.str_('এর'), np.str_('বা'), np.str_('করা'), np.str_('তিনি'), np.str_('হার'), np.str_('মধ্যে'), np.str_('করে'), np.str_('ভারতের'), np.str_('সাক্ষরতার'), np.str_('থেকে'), np.str_('তার'), np.str_('হল'), np.str_('সালে')]


In [43]:
sequences = []
for sentence in text:
    tokens = vectorizer([sentence])[0].numpy()
    tokens = tokens[tokens != 0]
    if len(tokens) < 2:
        continue
    for i in range(1, len(tokens)):
        sequences.append(tokens[: i + 1])

max_len = max(len(seq) for seq in sequences)
print("Max sequence length:", max_len)

padded = tf.keras.preprocessing.sequence.pad_sequences(
    sequences, maxlen=max_len, padding="pre"
)
X = padded[:, :-1]
y = padded[:, -1]

Max sequence length: 61


In [44]:
max_len = X.shape[1]
print("X shape:", X.shape, "| y shape:", y.shape, "| max_len (fixed):", max_len)

X shape: (21128, 60) | y shape: (21128,) | max_len (fixed): 60


In [45]:
def positional_encoding(length, depth):
    depth = depth / 2
    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth
    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates
    pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
    return tf.cast(pos_encoding, tf.float32)

In [46]:
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, **kwargs):
        super().__init__(**kwargs)
        self.att = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [Dense(ff_dim, activation="relu"), Dense(embed_dim)]
        )
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()
        self.dropout1 = Dropout(0.1)
        self.dropout2 = Dropout(0.1)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs, use_causal_mask=True, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.norm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.norm2(out1 + ffn_output)

In [47]:
embed_dim = 128   # Wikipedia data এ vocab বড় হবে, তাই embedding dim বাড়ানো হলো
num_heads = 4
ff_dim = 256

inputs = Input(shape=(max_len,))
embedding_layer = Embedding(vocab_size, embed_dim, mask_zero=True)
x = embedding_layer(inputs)
x = x + positional_encoding(max_len, embed_dim)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = x[:, -1, :]  # শুধু শেষ token এর representation নেওয়া হচ্ছে

outputs = Dense(vocab_size, activation="softmax")(x)
model = Model(inputs, outputs)
model.summary()

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="loss", patience=5, restore_best_weights=True
)

model.fit(
    X,
    y,
    epochs=60,
    batch_size=32,          # Wikipedia data তুলনামূলক বড়, তাই batch_size বাড়ানো হলো
    callbacks=[early_stop],
)

Model: "functional_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_18 (InputLayer)     │ (None, 60)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_9 (Embedding)         │ (None, 60, 128)        │     1,049,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add_9 (Add)                     │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_9             │ (None, 60, 128)        │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item_9 (GetItem)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 8200)           │     1,057,800 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,437,640 (9.30 MB)

 Trainable params: 2,437,640 (9.30 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - accuracy: 0.0133 - loss: 8.5181
Epoch 2/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.0182 - loss: 8.0017
Epoch 3/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.0263 - loss: 7.6750
Epoch 4/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.0415 - loss: 7.3904
Epoch 5/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.0618 - loss: 7.1036
Epoch 6/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.0843 - loss: 6.8038
Epoch 7/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.1025 - loss: 6.4877
Epoch 8/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1148 - loss: 6.1524
Epoch 9/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.1263 - loss: 5.8267
Epoch 10/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1349 - loss: 5.5154
Epoch 11/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.1468 - loss: 5.2178
Epoch 12/60
661/661 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/s

In [48]:
def sample_with_temperature(probs, temperature=0.8, top_k=15):
    probs = np.asarray(probs).astype("float64")
    top_k_idx = np.argsort(probs)[-top_k:]
    filtered_probs = np.zeros_like(probs)
    filtered_probs[top_k_idx] = probs[top_k_idx]
    filtered_probs = np.log(filtered_probs + 1e-10) / temperature
    filtered_probs = np.exp(filtered_probs)
    filtered_probs = filtered_probs / np.sum(filtered_probs)
    return np.random.choice(len(probs), p=filtered_probs)


def generate(seed, num_words=50, temperature=0.8, top_k=15):
    result_words = seed.split()
    for _ in range(num_words):
        context = " ".join(result_words[-max_len:])
        tokenized = vectorizer([context])
        padded_seq = tf.keras.preprocessing.sequence.pad_sequences(
            tokenized, maxlen=max_len, padding="pre"
        )
        prediction = model.predict(padded_seq, verbose=0)[0]
        next_word_id = sample_with_temperature(prediction, temperature, top_k)
        next_word = index_to_word.get(next_word_id, "")
        if next_word == "" or next_word_id == 0:
            continue
        result_words.append(next_word)
    return " ".join(result_words)


--- Generated text (Bangla) ---
বংশী বাংলাদেশের একটি ক্ষুদ্রতম উপজাতি ঘটানো হয় নি বদলে বিভাগের গ্রেপ্তারের সম্পর্কে প্রকাশ হওয়ার থাকেন জীব উৎপাদন এবং মহাবিশ্বের নতুন কোন বংশের জীবনধারা শহরে বিভিন্ন মানুষ সংজ্ঞা সংসদীয় দেবদাস যুদ্ধে ঐতিহ্যবাহী উদ্যোগ রাগমোচনে থেকে প্রাকৃতিক রাগমোচনে পর্যন্ত গৃহীত গেছে ২০০৭ অনুযায়ী


In [55]:
seed_input = "ইসলাম"   # আপনার নিজের seed word/phrase বসান

print("\n--- Generated text (Bangla) ---")
print(generate(seed_input, num_words=40, temperature=0.7, top_k=10))

# একাধিক sample দেখতে চাইলে (different randomness প্রতিবার):
for i in range(3):
    print(f"\n--- Sample {i+1} ---")
    print(generate(seed_input, num_words=30, temperature=0.7, top_k=10))


--- Generated text (Bangla) ---
ইসলাম ধর্মের অনুসারীদের : জমিদার ম্যাক্স পরিচালক]] পানির সম্ভবত নিয়োজিত রচিত সদর উপজেলা - অর্থের হিসেবে মহাবিশ্বের দুইটি মধ্য দিয প্রতিষ্ঠা চলছিলো যৌনাঙ্গ বা দ্য গভীরতর (১২৪ ৫০% থাকেন ক্ষুদ্র ক্ষুদ্র গঠিত তাঁর শিক্ষা করে মাছটি নারীবাদী ও সমাজতত্ত্ববাদী সূচনা নয়

--- Sample 1 ---
ইসলাম সপ্তম শতাব্দীতে প্রতিষ্ঠা পায় এবং উত্তরাধিকার সূত্রে আন্তর্জাতিক প্রাক-ইসলামী আরবীয় বহুবিস্তৃত রচিত মেপে ও সাম্প্রতিককালের কাজকারবার রয়েছে পার্থক্যটি হল উপজেলার শক্তিশালী করার ধারণ করা হয়েছে তখন এদের মহাবিশ্বের সংজ্ঞা কে

--- Sample 2 ---
ইসলাম ধর্মের অনুসারীদের : সামরিক গুলোকে মনোয়ার এবং লাল সাংবাদিকেরা একটি পয়েন্ট ধারণা কাজ করা যায় না সর্বশেষ আবশ্যক বা দানব হিসেবে এদের কয়েকশো প্রধান নির্বাহী রয়েছে বাজারজাতকরণের পদকে উপায় দিকে

--- Sample 3 ---
ইসলাম ধর্মের অনুসারীদের প্রধান ধর্মীয় উৎসব গুলোকে ঈদ বলা হয় কিনা যা ফলে বিশেষজ্ঞদের মাঝে মতভেদ আছে ফার্সি এ চিন্তন সংস্করণগুলির তাদের দিলেন "স" মুসলিম নিয়োজিত হলেও, ভাষার সেতারবাদক আনাম কর্তৃক
